# Model Evaluation — Entity Resolution Pipeline

Loads the committed 50-row sample dataset, runs ingestion -> preprocessing -> blocking ->
resolution, then reports:

- **Blocking metrics**: reduction ratio, pair completeness (vs. the sample `ground_truth.csv`)
- **Resolution metrics**: precision / recall / F1 at the candidate-pair level, auto-link rate, review rate

Run from the `backend/` directory with the project's virtualenv active and
`DATABASE_URL` / `TEST_DATABASE_URL` set (see `.env.example`).

In [ ]:
import sys
sys.path.insert(0, "../backend")

from pathlib import Path
from sqlalchemy.orm import sessionmaker

from app.core.db import Base, engine
from app.services.ingestion_service import IngestionService
from app.services.preprocessing_service import PreprocessingService
from app.services.resolution_service import ResolutionService
from app.resolution.blocking import PhoneticVillageBlocker, Record
from app.resolution.evaluation import evaluate_resolution, load_ground_truth
from app.models.clean_record import CleanRecord

Session = sessionmaker(bind=engine)
SAMPLE_DIR = Path("../data/synthetic/samples")

In [ ]:
Base.metadata.create_all(bind=engine)
db = Session()

IngestionService(db).ingest_all(SAMPLE_DIR)
PreprocessingService(db).run()
summary = ResolutionService(db).run()
summary

## Blocking metrics

In [ ]:
clean_rows = db.query(CleanRecord).all()
records = [
    Record(
        id=r.id, name_norm=r.name_norm, father_norm=r.father_norm,
        name_phonetic=r.name_phonetic, mobile10=r.mobile10, village_code=r.village_code,
        district_code=r.district_code, survey_no=r.survey_no, dob=r.dob,
        bank_last4=r.bank_last4, crop=r.crop, season=r.season,
    )
    for r in clean_rows
]
pairs = PhoneticVillageBlocker().candidate_pairs(records)
n = len(records)
max_pairs = n * (n - 1) / 2
reduction_ratio = 1 - (len(pairs) / max_pairs) if max_pairs else 0.0
print(f"records={n} candidate_pairs={len(pairs)} reduction_ratio={reduction_ratio:.3f}")

## Resolution metrics (precision / recall / F1 vs. ground truth)

In [ ]:
metrics = evaluate_resolution(db)
print(f"precision={metrics.precision:.3f}  recall={metrics.recall:.3f}  f1={metrics.f1:.3f}")
print(f"auto_link_rate={metrics.auto_link_rate:.3f}  review_rate={metrics.review_rate:.3f}")
ground_truth_rows = len(load_ground_truth())
print(f"ground_truth rows loaded: {ground_truth_rows}")

In [ ]:
db.close()